# SFT：一个样本如何变成一次 LoRA 更新

**适合读者**：已经跑过 MiniGPT 前向，准备学习 SFT/LoRA 的开发者与算法工程师。

**先修**：next-token prediction、交叉熵、PyTorch 基础；不需要下载模型或使用 GPU。

**学习目标**：沿着 template → token IDs → assistant-only labels → loss → backward → adapter → 独立重载 → held-out comparison 走完一条可执行链路，并说明每一步没有证明什么。


## 实验路线

1. 固定对话模板并看见监督边界。
2. 用随机初始化 MiniGPT 作为冻结基座，只给输出头挂 LoRA。
3. 让一个样本的 assistant-only loss 下降。
4. 在全新基座实例上重载 adapter。
5. 单独报告 held-out loss，不拿训练样本冒充质量评测。

> 这是机制教程，不是有用模型训练。真实 checkpoint 必须使用它自己的 tokenizer、chat template、target modules 和数据治理流程。


In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

import torch

from about_llm.finetuning.lora import LoRALinear
from about_llm.from_scratch import ByteTokenizer, GPTConfig, MiniGPT

SEED = 21
torch.manual_seed(SEED)
tokenizer = ByteTokenizer()
print('torch:', torch.__version__, 'device: cpu', 'seed:', SEED)


## 1. Template 决定 token，也决定哪里有 loss

先预测：如果把 system、user 和 `assistant:` 前缀也设成 labels，模型究竟在学“回答”，还是在复述整段模板？

这里用透明的 byte tokenizer 和手写模板。目标 token 位于原序列的 assistant 答案区；因为语言模型的 target 向左移动一位，label mask 也必须按被预测 token 的原位置移动。


In [ ]:
SYSTEM = '回答用户问题。'

def encode_sft_example(
    user: str,
    assistant: str,
    *,
    assistant_only: bool = True,
) -> tuple[torch.Tensor, torch.Tensor, dict[str, int]]:
    prefix = f'system: {SYSTEM}\nuser: {user}\nassistant: '
    serialized = prefix + assistant
    token_ids = tokenizer.encode(serialized)
    answer_start = len(tokenizer.encode(prefix))

    input_ids = torch.tensor([token_ids[:-1]], dtype=torch.long)
    next_tokens = torch.tensor([token_ids[1:]], dtype=torch.long)
    original_target_positions = torch.arange(1, len(token_ids))
    supervised = (
        original_target_positions >= answer_start
        if assistant_only
        else torch.ones_like(original_target_positions, dtype=torch.bool)
    )
    labels = torch.where(supervised.unsqueeze(0), next_tokens, -100)
    metadata = {
        'serialized_tokens': len(token_ids),
        'answer_start': answer_start,
        'supervised_tokens': int(supervised.sum()),
    }
    return input_ids, labels, metadata

TRAIN_USER = 'KV Cache 保存什么？'
TRAIN_ASSISTANT = 'key 和 value'
train_input, train_labels, train_meta = encode_sft_example(
    TRAIN_USER,
    TRAIN_ASSISTANT,
)
print(train_meta)
print('input shape:', tuple(train_input.shape))
print('first supervised label index:', int((train_labels[0] != -100).nonzero()[0]))
assert train_meta['supervised_tokens'] == len(tokenizer.encode(TRAIN_ASSISTANT))
assert torch.all(train_labels[0, : train_meta['answer_start'] - 1] == -100)


## 2. 冻结基座，只训练低秩增量

MiniGPT 是随机初始化的教学模型。我们先保存未挂 adapter 的 logits，再把 tied LM head 包装成 `LoRALinear`。LoRA 的 B 矩阵从零开始，因此初始函数必须完全不变；这比“shape 能对上”更强。


In [ ]:
config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    context_length=128,
    model_dim=32,
    num_heads=4,
    num_layers=1,
    mlp_ratio=2,
)
model = MiniGPT(config).eval()
with torch.no_grad():
    base_logits, _ = model(train_input)

for parameter in model.parameters():
    parameter.requires_grad = False
model.lm_head = LoRALinear(model.lm_head, rank=4, alpha=8)
with torch.no_grad():
    wrapped_logits, _ = model(train_input)
torch.testing.assert_close(wrapped_logits, base_logits, rtol=0, atol=0)

trainable = {
    name: parameter.numel()
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
}
frozen_base_before = model.lm_head.base.weight.detach().clone()
print('trainable tensors:', trainable)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
print('trainable / total:', sum(trainable.values()), '/', total_parameters)


## 3. Backward 只说明优化器能拟合这个样本

运行前预测：第一个 backward 时，零初始化的 B 和随机初始化的 A 是否都会立刻得到非零梯度？答案是 B 先接到信号；当 B 不再为零后，A 才能收到有效梯度。

下面用同一个样本做 40 次更新。我们要求 loss 下降、梯度有限、冻结基座逐元素不变；但不把它解释成泛化或语言能力提升。


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=0.05)
losses = []
for step in range(40):
    optimizer.zero_grad(set_to_none=True)
    _, loss = model(train_input, train_labels)
    assert loss is not None and torch.isfinite(loss)
    loss.backward()
    if step == 0:
        print('step-0 |grad A|:', float(model.lm_head.lora_a.grad.abs().sum()))
        print('step-0 |grad B|:', float(model.lm_head.lora_b.grad.abs().sum()))
    optimizer.step()
    losses.append(float(loss.detach()))

torch.testing.assert_close(model.lm_head.base.weight, frozen_base_before, rtol=0, atol=0)
assert losses[-1] < losses[0]
print('training loss:', round(losses[0], 4), '->', round(losses[-1], 4))


## 4. Adapter 必须能在全新基座实例上重载

只在当前 Python 对象里继续推理，无法发现 base identity、rank、alpha 或保存字段遗漏。这里把 adapter-only state 写入临时目录，再用相同 seed 构造全新基座并加载；临时文件在 cell 结束后自动删除。


In [ ]:
model.eval()
with torch.no_grad():
    trained_logits, _ = model(train_input)

with tempfile.TemporaryDirectory() as temporary_directory:
    adapter_path = Path(temporary_directory) / 'adapter.pt'
    torch.save(model.lm_head.adapter_state_dict(), adapter_path)
    adapter = torch.load(adapter_path, map_location='cpu', weights_only=True)

torch.manual_seed(SEED)
reloaded = MiniGPT(config).eval()
reloaded.lm_head = LoRALinear(
    reloaded.lm_head,
    rank=int(adapter['rank']),
    alpha=float(adapter['alpha']),
)
with torch.no_grad():
    reloaded.lm_head.lora_a.copy_(adapter['lora_a'])
    reloaded.lm_head.lora_b.copy_(adapter['lora_b'])
    reloaded_logits, _ = reloaded(train_input)
torch.testing.assert_close(reloaded_logits, trained_logits, rtol=0, atol=0)
print('saved keys:', sorted(adapter))
print('independent reload max error:', float((reloaded_logits - trained_logits).abs().max()))


## 5. Held-out 是独立问题，不是训练 loss 的别名

最后用一个未参加更新的问题比较同一冻结基座与 adapter。结果可能改善，也可能退化；这个单例只教你把 evaluation 放在训练链路之外，不能支持发布判断。真实项目还需要固定 held-out artifact、切片、基线、解码配置和统计门禁。


In [ ]:
held_input, held_labels, held_meta = encode_sft_example(
    'LoRA 更新的是哪部分？',
    '低秩 adapter',
)
torch.manual_seed(SEED)
held_base = MiniGPT(config).eval()
with torch.no_grad():
    _, base_held_loss = held_base(held_input, held_labels)
    _, adapter_held_loss = reloaded(held_input, held_labels)
assert base_held_loss is not None and adapter_held_loss is not None
print('held-out supervised tokens:', held_meta['supervised_tokens'])
print('base held-out loss:', round(float(base_held_loss), 4))
print('adapter held-out loss:', round(float(adapter_held_loss), 4))


## 练习：故意监督整段模板

先预测下面两个数的关系，再运行答案脚手架：

1. assistant-only 与 all-token 的监督 token 数谁更大？
2. all-token loss 更低时，能否说明回答质量更高？
3. 只改变 rank、alpha 或训练样本中的一个变量，记录 train/held-out 两列，不要同时改。


In [ ]:
# 答案脚手架: all-token labels 会让模板与用户输入也参与目标。
all_token_input, all_token_labels, _ = encode_sft_example(
    TRAIN_USER,
    TRAIN_ASSISTANT,
    assistant_only=False,
)
torch.testing.assert_close(all_token_input, train_input, rtol=0, atol=0)
assistant_only_count = int((train_labels != -100).sum())
all_token_count = int((all_token_labels != -100).sum())
print({'assistant_only': assistant_only_count, 'all_token': all_token_count})
assert all_token_count > assistant_only_count


## 结论、常见坑与下一步

本实验实际证明了：shift 后的 assistant-only labels 可见；LoRA 初始函数与基座一致；只有 adapter 参数更新；一个训练样本 loss 下降；adapter-only state 能在相同基座 identity 上独立重载；held-out 结果被单独报告。

它没有证明：chat template 适配真实 checkpoint、训练数据合规、目标模块选择正确、模型泛化、单卡显存可用、QLoRA 数值稳定或 adapter 可发布。

最常见的坑是把 target shift 做错一位、把 padding/system/user 也监督、只保存 adapter 却忘记 base identity，以及用训练 loss 代替 held-out 质量。下一步进入 `projects/single-gpu-finetuning/`，先做目标 tokenizer/template preflight，再运行真实 PEFT/TRL 路径。
